In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

# 1. Resolve Project Root Directory
CURRENT_DIR = os.getcwd()
PROJECT_DIR = os.path.dirname(CURRENT_DIR) if os.path.basename(CURRENT_DIR) == 'jupyter_notebooks' else CURRENT_DIR

# 2. Set Input and Output Directories
INPUTS_DIR = os.path.join(PROJECT_DIR, 'inputs', 'cherry_leaves')
TRAIN_DIR = os.path.join(INPUTS_DIR, 'train')
VAL_DIR = os.path.join(INPUTS_DIR, 'validation')
TEST_DIR = os.path.join(INPUTS_DIR, 'test')

OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs', 'v1')
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Image & Training Hyperparameters
IMAGE_WIDTH, IMAGE_HEIGHT = 256, 256
IMAGE_SHAPE = (IMAGE_WIDTH, IMAGE_HEIGHT, 3)
BATCH_SIZE = 32
EPOCHS = 25

print(f"TensorFlow Version: {tf.__version__}")
print(f"Train Directory: {TRAIN_DIR}")
print(f"Outputs Directory: {OUTPUTS_DIR}")

In [ ]:
# Train generator with data augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation and Test generators (only rescaling, no augmentation)
val_test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

# Create directory iterators
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

# Save class indices map
class_indices = train_generator.class_indices
print(f"\nClass Indices: {class_indices}")

with open(os.path.join(OUTPUTS_DIR, 'class_indices.pkl'), 'wb') as f:
    pickle.dump(class_indices, f)

In [ ]:
def create_cnn_model(input_shape):
    """
    Builds a custom CNN model architecture optimized for binary leaf classification.
    """
    model = Sequential([
        # First Convolutional Block
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D((2, 2)),

        # Second Convolutional Block
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),

        # Third Convolutional Block
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),

        # Fully Connected Layers
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')  # Binary classification output
    ])
    
    return model

model = create_cnn_model(IMAGE_SHAPE)
model.summary()

In [ ]:
# Compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Define callbacks
model_save_path = os.path.join(OUTPUTS_DIR, 'powdery_mildew_detector_model.h5')

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=model_save_path,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

print("Model compiled and callbacks configured.")